# W5 — LoRA Fine-Tune (Gemma-4 E2B, MLX)

Distill the **envelope -> script** capability into the local student via LoRA, then
measure lift against the W4 **new-pipeline baseline**.

- **Student / base model:** `models/gemma-4-e2b-it-bf16` (same model the baseline measured).
- **Trainer:** `mlx_vlm.lora` (SFT) — text-only LoRA; the model is multimodal but we train
  the language path only (no `--train-vision`).
- **Data:** `data/processed/{train,val,test}.jsonl` (650 / 80 / 70), already in the
  `{"messages": [...]}` chat format mlx_vlm expects — **no conversion needed**.
- **Fallback (documented):** if MLX OOMs / fails, the plan is **Colab + PyTorch + Unsloth**
  (see the last section). `unsloth`/`torch` are intentionally absent locally.

> ⚠️ **Training is gated OFF in this notebook** (`RUN_TRAINING = False`). Running every cell
> here does setup + prints the exact command; it does **not** start training. Kick off the
> run yourself from a terminal (it's long-lived).

## 1. Environment smoke-test (the W5 prep gate)

Confirm the MLX training stack imports before committing to the local path. If `mlx_vlm`
is missing or errors, stop and use the Colab fallback.

In [1]:
import sys, importlib.util as u, importlib.metadata as md
from pathlib import Path


def _root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / 'pyproject.toml').exists():
            return c
    raise RuntimeError('project root not found')


PROJECT_ROOT = _root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REQUIRED = ['mlx', 'mlx_lm', 'mlx_vlm']
FALLBACK = ['unsloth', 'torch']  # only needed for the Colab path
print('--- MLX path (local) ---')
ok = True
for m in REQUIRED:
    spec = u.find_spec(m)
    ver = md.version(m.replace('_', '-')) if spec else ''
    print(f'  {m:10} {"OK" if spec else "MISSING":8} {ver}')
    ok = ok and spec is not None
print('  mlx_vlm.lora     :', 'OK' if u.find_spec('mlx_vlm.lora') else 'MISSING')
print('--- Colab fallback stack ---')
for m in FALLBACK:
    print(f'  {m:10} {"present" if u.find_spec(m) else "absent (expected locally)"}')
print('\nMLX local training path:', 'READY' if ok else 'NOT available -> use Colab fallback')

--- MLX path (local) ---
  mlx        OK       0.31.2
  mlx_lm     OK       0.31.3
  mlx_vlm    OK       0.5.0


  mlx_vlm.lora     : OK
--- Colab fallback stack ---
  unsloth    absent (expected locally)
  torch      absent (expected locally)

MLX local training path: READY


## 2. Data check

The processed splits load straight into a 🤗 `datasets` dataset with a `messages` column.
mlx_vlm's SFT trainer reads `messages` and treats images as optional (none here = text-only).

In [2]:
from datasets import load_dataset

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
ds = {s: load_dataset(str(DATA_DIR), split=split)
      for s, split in [('train', 'train'), ('val', 'validation'), ('test', 'test')]}
for s, d in ds.items():
    print(f'{s:5} {len(d):4} rows  cols={d.column_names}')

ex = ds['train'][0]['messages']
print('\nroles:', [m['role'] for m in ex])
print('user[:160]      :', ex[0]['content'][:160].replace(chr(10), ' '))
print('assistant[:160] :', ex[-1]['content'][:160].replace(chr(10), ' '))

train  650 rows  cols=['messages']
val     80 rows  cols=['messages']
test    70 rows  cols=['messages']

roles: ['user', 'assistant']
user[:160]      : Metadata envelope: ```json {   "format": "csv",   "file_size_bytes": 195,   "encoding": "utf-8-sig",   "schema_version": "0.1",   "schema": {     "delimiter": "
assistant[:160] : <analysis>Source: a flat CSV (utf-8-sig, so a BOM may prefix the header) with 5 columns and one row per order. The same user can appear on multiple rows (a repe


## 3. The 'before' target — new-pipeline baseline

What the **un-fine-tuned** student scores on the same task + same held-out test set.
Fine-tuning has to move **schema_compliance** and **content_accuracy** (and fix uc1).

In [3]:
import json, glob

runs = sorted(glob.glob(str(PROJECT_ROOT / 'results' / 'baseline_newpipeline_gemma_*' / 'summary.json')))
BASELINE = json.load(open(runs[-1])) if runs else None
if BASELINE:
    a = BASELINE['aggregate']
    print('baseline run:', BASELINE['run_id'])
    print('overall :', {k: round(v, 3) for k, v in a['overall'].items()})
    print(f"all-pass : {a['n_accepted']}/{a['n_cases']}")
    print('per use case (ca):')
    for uc, b in a['by_use_case'].items():
        print(f"  {uc:26} ca={b['content_accuracy']:.2f}  n={b['n']}")
else:
    print('no baseline summary found — run scripts/run_pipeline_baseline.py first')

baseline run: baseline_newpipeline_gemma_2026-05-31_193850
overall : {'format_validity': 0.9, 'schema_compliance': 0.771, 'loadability': 0.9, 'content_accuracy': 0.657}
all-pass : 41/70
per use case (ca):
  uc1_csv_to_json_nested     ca=0.11  n=14
  uc2_json_to_csv_flatten    ca=0.94  n=8
  uc3_txt_log_to_csv         ca=0.43  n=14
  uc4_csv_to_txt_report      ca=0.84  n=19
  uc5_schema_migration       ca=1.00  n=15


## 4. LoRA configuration

Starting hyperparameters for ~650 training examples. These are reasonable defaults, not
tuned — adjust after the first run. `max_seq_length=3072` covers the longest example
(W4 proxy max ≈ 2094 'tokens' at chars/4; real tokenizer runs a bit higher, so we pad the
budget rather than risk truncation). `train_on_completions` trains on the assistant tokens
(analysis + script) only.

In [4]:
from pathlib import Path

CFG = {
    'model_path': 'models/gemma-4-e2b-it-bf16',
    'dataset': 'data/processed',     # HF load_dataset(dir) -> train/validation/test splits
    'split': 'train',
    'output_path': 'models/lora_gemma4e2b_scriptgen',
    'lora_rank': 8,
    'lora_alpha': 16,
    'lora_dropout': 0.05,
    'learning_rate': 1e-4,
    'batch_size': 1,
    'gradient_accumulation_steps': 4,   # effective batch 4, memory-safe on E2B
    'epochs': 3,
    'max_seq_length': 3072,
    'steps_per_report': 20,
    'steps_per_eval': 200,
    'steps_per_save': 200,
}
for k, v in CFG.items():
    print(f'{k:28} {v}')

model_path                   models/gemma-4-e2b-it-bf16
dataset                      data/processed
split                        train
output_path                  models/lora_gemma4e2b_scriptgen
lora_rank                    8
lora_alpha                   16
lora_dropout                 0.05
learning_rate                0.0001
batch_size                   1
gradient_accumulation_steps  4
epochs                       3
max_seq_length               3072
steps_per_report             20
steps_per_eval               200
steps_per_save               200


## 5. Training command — **run this yourself in a terminal**

The cell below *builds and prints* the exact `mlx_vlm.lora` command from the config. It does
**not** run it (`RUN_TRAINING = False`). Copy it into a terminal with the venv active:
`source .venv/bin/activate` then paste. Training is long-lived; let it run there, not here.

In [5]:
RUN_TRAINING = False   # <- flip to True only if you really want to train from the notebook

cmd = [
    'python', '-m', 'mlx_vlm.lora',
    '--model-path', CFG['model_path'],
    '--dataset', CFG['dataset'],
    '--split', CFG['split'],
    '--train-mode', 'sft',
    '--train-on-completions',
    '--lora-rank', str(CFG['lora_rank']),
    '--lora-alpha', str(CFG['lora_alpha']),
    '--lora-dropout', str(CFG['lora_dropout']),
    '--learning-rate', str(CFG['learning_rate']),
    '--batch-size', str(CFG['batch_size']),
    '--gradient-accumulation-steps', str(CFG['gradient_accumulation_steps']),
    '--epochs', str(CFG['epochs']),
    '--max-seq-length', str(CFG['max_seq_length']),
    '--steps-per-report', str(CFG['steps_per_report']),
    '--steps-per-eval', str(CFG['steps_per_eval']),
    '--steps-per-save', str(CFG['steps_per_save']),
    '--output-path', CFG['output_path'],
]
# group each --flag with its value on one line for readable copy-paste
lines, i = ['python -m mlx_vlm.lora'], 3
while i < len(cmd):
    tok = cmd[i]
    if tok.startswith('--') and i + 1 < len(cmd) and not cmd[i + 1].startswith('--'):
        lines.append(f'{tok} {cmd[i + 1]}'); i += 2
    else:
        lines.append(tok); i += 1
printable = ' \\\n    '.join(lines)
print('# run from the project root, venv active:\n')
print(printable)

if RUN_TRAINING:
    import subprocess
    print('\n[RUN_TRAINING=True] launching training in-process...')
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)
else:
    print('\n[RUN_TRAINING=False] not training. Copy the command above into a terminal.')

# run from the project root, venv active:

python -m mlx_vlm.lora \
    --model-path models/gemma-4-e2b-it-bf16 \
    --dataset data/processed \
    --split train \
    --train-mode sft \
    --train-on-completions \
    --lora-rank 8 \
    --lora-alpha 16 \
    --lora-dropout 0.05 \
    --learning-rate 0.0001 \
    --batch-size 1 \
    --gradient-accumulation-steps 4 \
    --epochs 3 \
    --max-seq-length 3072 \
    --steps-per-report 20 \
    --steps-per-eval 200 \
    --steps-per-save 200 \
    --output-path models/lora_gemma4e2b_scriptgen

[RUN_TRAINING=False] not training. Copy the command above into a terminal.


## 6. After training — measure the lift (gated)

Once `models/lora_gemma4e2b_scriptgen/adapters.safetensors` exists, re-run the **same**
new-pipeline baseline with the adapter loaded, then diff against section 3.

Wiring note: inference goes through `src/models/gemma_mlx.py`, which currently loads the
base model with no adapter. To eval the fine-tuned student, load with the adapter (mlx_vlm
`load(..., adapter_path=...)` / `apply_lora_layers`). That small change + an `--adapter`
flag on `scripts/run_pipeline_baseline.py` is the follow-up after the first successful run.

In [6]:
RUN_EVAL = False   # <- flip to True after training completes and the adapter exists

ADAPTER = PROJECT_ROOT / CFG['output_path'] / 'adapters.safetensors'
print('adapter present:', ADAPTER.exists(), '->', ADAPTER)

if RUN_EVAL and ADAPTER.exists():
    # Re-run the held-out test eval with the adapter, then compare to BASELINE['aggregate'].
    # (Requires the gemma_mlx adapter-loading follow-up described above.)
    print('TODO: run scripts/run_pipeline_baseline.py --adapter', ADAPTER,
          'and diff overall fv/sc/ld/ca vs section 3.')
else:
    print('[RUN_EVAL=False or no adapter] eval skipped — train first.')

adapter present: False -> /Users/thatt/Dev/AI project/data-morph/models/lora_gemma4e2b_scriptgen/adapters.safetensors
[RUN_EVAL=False or no adapter] eval skipped — train first.


## 7. Colab fallback (if local MLX fails)

If section 1 reports the MLX path NOT ready, or training OOMs / errors on this machine, the
documented fallback (per `CLAUDE.md` risk register) is **Google Colab + PyTorch + Unsloth**:

1. Upload `data/processed/{train,val}.jsonl` to the Colab session.
2. `pip install unsloth` (CUDA wheel) and load a Gemma base via Unsloth's `FastModel`.
3. LoRA-fine-tune on the same `messages` chat format (Unsloth's `SFTTrainer` + chat template).
4. Download the adapter, convert to MLX if running inference locally, and eval as in section 6.

Keep the **same data, same LoRA rank, and same held-out test set** so the before/after
comparison stays valid regardless of which backend trains the adapter.